# DistilBERT — 200k

This notebook contains the direct training code for DistilBERT. It is one experiment only, so the examiner can read it from top to bottom.

The training cell is enabled. Run this notebook individually when reproducing the model. Existing results are kept in results; a new run writes to reproduced_runs.

Approximate historical training time:

- DistilBERT 200k: about 52 minutes; BERT-base 200k: about 1 hour 40 minutes; HateBERT 200k: about 1 hour 41 minutes.
- DistilBERT full: about 3 hours 8 minutes; BERT-base full: about 6 hours; HateBERT full: about 6 hours.

These times depend on the available GPU. Do not use Run All across the Transformer notebooks.

Settings:

- train / validation / test: 200,000 / 20,000 / 178,083
- epochs: 2
- max length: 128
- train / evaluation batch size: 16 / 32
- learning rate: 0.00002
- weight decay: 0.01
- warmup ratio: 0.0
- evaluation / saving: epoch / epoch
- logging: every 200 steps
- best model metric: toxic_f1
- FP16: True
- full-data historical extras: not used
- seed: 42

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)
REPRODUCED = ROOT / "reproduced_runs"


MODEL_KEY = "distilbert"
MODEL_NAME = "distilbert-base-uncased"
MODEL_LABEL = "DistilBERT"
EXPERIMENT = "200k"
SETTINGS = {'seed': 42, 'epochs': 2, 'max_length': 128, 'train_batch_size': 16, 'eval_batch_size': 32, 'learning_rate': 2e-05, 'weight_decay': 0.01, 'warmup_ratio': 0.0, 'evaluation_strategy': 'epoch', 'save_strategy': 'epoch', 'logging_strategy': 'steps', 'logging_steps': 200, 'save_total_limit': 2, 'load_best_model_at_end': True, 'metric_for_best_model': 'toxic_f1', 'greater_is_better': True, 'fp16': True}
DATA = {
    "train": ROOT / "data/splits/200k/train.csv",
    "validation": ROOT / "data/splits/200k/validation.csv",
    "test": ROOT / "data/splits/full/test.csv",
}
OUTPUT_DIR = REPRODUCED / EXPERIMENT / MODEL_KEY

TRAIN_ROWS = 200000
VALIDATION_ROWS = 20000

## Load the recorded data

In [4]:
train = pd.read_csv(DATA["train"])
validation = pd.read_csv(DATA["validation"])
test = pd.read_csv(DATA["test"])
assert len(train) == TRAIN_ROWS
assert len(validation) == VALIDATION_ROWS
assert len(test) == 178083
assert train["target"].ge(0.5).astype("int8").equals(train["label"].astype("int8"))
assert validation["target"].ge(0.5).astype("int8").equals(validation["label"].astype("int8"))
assert test["target"].ge(0.5).astype("int8").equals(test["label"].astype("int8"))
print("Train:", len(train), "Validation:", len(validation), "Test:", len(test))

Train: 200000 Validation: 20000 Test: 178083


## Core training code

In [6]:
RUN_TRAINING = True

if RUN_TRAINING:
    import json
    import numpy as np
    import torch
    from sklearn.metrics import accuracy_score, f1_score
    from torch.utils.data import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

    class CommentDataset(Dataset):
        def __init__(self, frame, tokenizer):
            self.frame = frame.reset_index(drop=True)
            self.tokenizer = tokenizer

        def __len__(self):
            return len(self.frame)

        def __getitem__(self, index):
            row = self.frame.iloc[index]
            item = self.tokenizer(
                str(row["comment_text"]),
                truncation=True,
                max_length=SETTINGS["max_length"],
            )
            item["labels"] = int(row["label"])
            return item

    def metrics(result):
        predicted = np.argmax(result.predictions, axis=1)
        return {
            "accuracy": accuracy_score(result.label_ids, predicted),
            "toxic_f1": f1_score(result.label_ids, predicted, zero_division=0),
        }

    if OUTPUT_DIR.exists():
        raise FileExistsError("This output directory already exists.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )
    train_set = CommentDataset(train, tokenizer)
    validation_set = CommentDataset(validation, tokenizer)
    test_set = CommentDataset(test, tokenizer)

    training_kwargs = {
        "output_dir": str(OUTPUT_DIR / "checkpoints"),
        "num_train_epochs": SETTINGS["epochs"],
        "per_device_train_batch_size": SETTINGS["train_batch_size"],
        "per_device_eval_batch_size": SETTINGS["eval_batch_size"],
        "learning_rate": SETTINGS["learning_rate"],
        "weight_decay": SETTINGS["weight_decay"],
        "warmup_ratio": SETTINGS["warmup_ratio"],
        "eval_strategy": SETTINGS["evaluation_strategy"],
        "save_strategy": SETTINGS["save_strategy"],
        "logging_strategy": SETTINGS["logging_strategy"],
        "logging_steps": SETTINGS["logging_steps"],
        "save_total_limit": SETTINGS["save_total_limit"],
        "load_best_model_at_end": SETTINGS["load_best_model_at_end"],
        "metric_for_best_model": SETTINGS["metric_for_best_model"],
        "greater_is_better": SETTINGS["greater_is_better"],
        "fp16": SETTINGS["fp16"],
        "seed": SETTINGS["seed"],
        "data_seed": SETTINGS["seed"],
        "report_to": [],
    }
    if EXPERIMENT == "full":
        training_kwargs.update({
            "lr_scheduler_type": SETTINGS["scheduler"],
            "dataloader_num_workers": SETTINGS["workers"],
            "dataloader_pin_memory": SETTINGS["pin_memory"],
            "optim": SETTINGS["optimizer"],
            "save_safetensors": True,
        })
    training_args = TrainingArguments(**training_kwargs)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_set,
        eval_dataset=validation_set,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=metrics,
    )
    trainer.train()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(OUTPUT_DIR / "best_model")
    tokenizer.save_pretrained(OUTPUT_DIR / "best_model")

    result = trainer.predict(test_set)
    logits = np.asarray(result.predictions)
    probabilities = torch.softmax(torch.tensor(logits), dim=1).numpy()
    predictions = test[["id", "comment_text", "target", "label"]].copy()
    predictions["predicted_label"] = np.argmax(logits, axis=1)
    predictions["toxic_probability"] = probabilities[:, 1]
    predictions["non_toxic_logit"] = logits[:, 0]
    predictions["toxic_logit"] = logits[:, 1]
    predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    (OUTPUT_DIR / "settings.json").write_text(json.dumps(SETTINGS, indent=2))